## Dataset Exploration

In [0]:
df = spark.table("resyn_27_k_2")
display(df)


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
display(null_counts)

In [0]:
df = spark.table("resyn_27_k_2")

df.printSchema()
df.select("Instruction", "Response").show(3, truncate=False)

## New Dataset Creation

In [0]:
from pyspark.sql.functions import col

clean_df = df.select(
    col("Instruction"),
    col("Response")[0].alias("VerilogCode")
)

clean_df.display()
print("rows =", clean_df.count())

In [0]:
clean_df.write.format("delta").mode("overwrite").saveAsTable("resyn27k_clean")

In [0]:
clean_df = spark.table("resyn27k_clean")

### Vector DF

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

vector_df = clean_df.withColumn(
    "id",
    monotonically_increasing_id()
)

In [0]:
from pyspark.sql.functions import concat_ws

vector_df = vector_df.withColumn(
    "embedding_text",
    col("Instruction")
)

vector_df.display()

In [0]:
vector_df.write.format("delta").mode("overwrite").saveAsTable("resyn27k_vector")

### KG Dataset

- Extract KG
- Create table with KG parameters
- (During search) - How to go from Original Instructions -> KG-based Search
  - Option 1: LLM parameter estimation -> KG-based Search
  - Option 2: Rule Based Parameter extraction -> KG-based Search

In [0]:
clean_df

In [0]:
display(clean_df.limit(5))
clean_df.printSchema()

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

kg_base = clean_df.withColumn("doc_id", monotonically_increasing_id())

In [0]:
from pyspark.sql.functions import lower, col

kg_base = kg_base.withColumn("code_lc", lower(col("VerilogCode")))

In [0]:
from pyspark.sql.functions import regexp_extract

kg_base = kg_base.withColumn(
    "module_name",
    regexp_extract(col("VerilogCode"), r"(?i)\bmodule\s+([a-zA-Z_]\w*)", 1)
)

In [0]:
from pyspark.sql.functions import instr, when, lit

def has(substr):
    return (instr(col("code_lc"), lit(substr)) > 0)

kg_feat = (kg_base
    .withColumn("has_always", (instr(col("code_lc"), "always") > 0).cast("int"))
    .withColumn("has_assign", (instr(col("code_lc"), "assign") > 0).cast("int"))
    .withColumn("has_case", (instr(col("code_lc"), "case") > 0).cast("int"))
    .withColumn("has_if", (instr(col("code_lc"), "if") > 0).cast("int"))
    .withColumn("has_for", (instr(col("code_lc"), "for") > 0).cast("int"))
    .withColumn("has_generate", (instr(col("code_lc"), "generate") > 0).cast("int"))
    .withColumn("has_posedge", (instr(col("code_lc"), "posedge") > 0).cast("int"))
    .withColumn("has_negedge", (instr(col("code_lc"), "negedge") > 0).cast("int"))
)

In [0]:
kg_feat = (kg_feat
    .withColumn("tag_fsm", (
        (instr(col("code_lc"), "state") > 0) &
        (instr(col("code_lc"), "case") > 0)
    ).cast("int"))
    .withColumn("tag_counter", (
        (instr(col("code_lc"), "+ 1") > 0) | (instr(col("code_lc"), "++") > 0) |
        (instr(col("code_lc"), "counter") > 0)
    ).cast("int"))
    .withColumn("tag_mux", (
        (instr(col("code_lc"), "case") > 0) | (instr(col("code_lc"), "?:") > 0)
    ).cast("int"))
    .withColumn("tag_alu", (instr(col("code_lc"), "alu") > 0).cast("int"))
    .withColumn("tag_shift", (
        (instr(col("code_lc"), "<<") > 0) | (instr(col("code_lc"), ">>") > 0) |
        (instr(col("code_lc"), "shift") > 0)
    ).cast("int"))
)

In [0]:
from pyspark.sql.functions import regexp_extract

# Detect presence of a packed/unpacked array like: reg [..] name [..];
kg_feat = kg_feat.withColumn(
    "tag_mem_array",
    (regexp_extract(col("VerilogCode"), r"(?is)\breg\b[^\n;]*\[[^\]]+\]\s*\w+\s*\[[^\]]+\]\s*;", 0) != "").cast("int")
)

kg_feat = (kg_feat
    .withColumn("tag_ram", (
        (instr(col("code_lc"), "ram") > 0) | (instr(col("code_lc"), "memory") > 0) | (col("tag_mem_array") == 1)
    ).cast("int"))
    .withColumn("tag_fifo", (
        (instr(col("code_lc"), "fifo") > 0) |
        ((instr(col("code_lc"), "wr_ptr") > 0) & (instr(col("code_lc"), "rd_ptr") > 0)) |
        ((instr(col("code_lc"), "write_ptr") > 0) & (instr(col("code_lc"), "read_ptr") > 0))
    ).cast("int"))
)

In [0]:
kg_feat = (kg_feat
    .withColumn("has_clk_signal", (
        (instr(col("code_lc"), " clk") > 0) | (instr(col("code_lc"), "(clk") > 0)
    ).cast("int"))
    .withColumn("has_reset_signal", (
        (instr(col("code_lc"), "reset") > 0) | (instr(col("code_lc"), "rst") > 0)
    ).cast("int"))
    .withColumn("tag_async_reset", (
        (instr(col("code_lc"), "negedge rst") > 0) |
        (instr(col("code_lc"), "negedge reset") > 0) |
        (instr(col("code_lc"), "or negedge") > 0)
    ).cast("int"))
)

In [0]:
kg_table = kg_feat.select(
    "doc_id",
    "module_name",
    "has_always","has_assign","has_case","has_if","has_for","has_generate",
    "has_posedge","has_negedge",
    "has_clk_signal","has_reset_signal","tag_async_reset",
    "tag_fsm","tag_counter","tag_mux","tag_alu","tag_shift",
    "tag_mem_array","tag_ram","tag_fifo"
)

display(kg_table.limit(20))

In [0]:
kg_table.write.format("delta").mode("overwrite").saveAsTable("resyn27k_kg_features")

In [0]:
kg_features = spark.table("resyn27k_kg_features")
display(kg_features.limit(10))

In [0]:
print("clean_df rows:", clean_df.count())
print("kg rows:", kg_features.count())

In [0]:
from pyspark.sql.functions import sum as spark_sum

display(kg_features.select(
    spark_sum("tag_fsm").alias("n_fsm"),
    spark_sum("tag_counter").alias("n_counter"),
    spark_sum("tag_fifo").alias("n_fifo"),
    spark_sum("tag_ram").alias("n_ram"),
    spark_sum("tag_shift").alias("n_shift")
))

In [0]:
kg_features.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("resyn27k_kg_features")

## Export 

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.export_volume")
print("Volume ready: workspace.default.export_volume")

In [0]:
kg_df = spark.table("resyn27k_kg_features")

kg_output_path = "/Volumes/workspace/default/export_volume/kg_features_parquet"

(
    kg_df
    .coalesce(1)   # remove this line if the table is large
    .write
    .mode("overwrite")
    .parquet(kg_output_path)
)

print(f"KG parquet written to: {kg_output_path}")

In [0]:
vec_df = spark.table("resyn27k_vector")

vec_output_path = "/Volumes/workspace/default/export_volume/vector_parquet"

(
    vec_df
    .coalesce(1)   # remove this line if the table is large
    .write
    .mode("overwrite")
    .parquet(vec_output_path)
)

print(f"Vector parquet written to: {vec_output_path}")

## KG V2

In [0]:
clean_df = spark.table("resyn27k_clean")
clean_df
display(clean_df.limit(5))
clean_df.printSchema()

from pyspark.sql.functions import (
    monotonically_increasing_id, lower, col, regexp_extract, regexp_replace,
    instr, lit, when, length, size, split, greatest
)

# -----------------------------------------------------------------------------
# Base table
# -----------------------------------------------------------------------------

kg_base = clean_df.withColumn("doc_id", monotonically_increasing_id())

kg_base = kg_base.withColumn("code_lc", lower(col("VerilogCode")))

kg_base = kg_base.withColumn(
    "module_name",
    regexp_extract(col("VerilogCode"), r"(?i)\bmodule\s+([a-zA-Z_]\w*)", 1)
)

# Strip comments to make heuristic matching less noisy
kg_base = kg_base.withColumn(
    "code_no_comments",
    regexp_replace(
        regexp_replace(col("code_lc"), r"(?s)/\*.*?\*/", " "),
        r"//.*",
        " "
    )
)

# -----------------------------------------------------------------------------
# Helper
# -----------------------------------------------------------------------------

def has(substr):
    return (instr(col("code_no_comments"), lit(substr)) > 0)

# -----------------------------------------------------------------------------
# Basic structural features
# -----------------------------------------------------------------------------

kg_feat = (kg_base
    .withColumn("has_always", has("always").cast("int"))
    .withColumn("has_assign", has("assign").cast("int"))
    .withColumn("has_case", has("case").cast("int"))
    .withColumn("has_if", has("if").cast("int"))
    .withColumn("has_for", has("for").cast("int"))
    .withColumn("has_generate", has("generate").cast("int"))
    .withColumn("has_posedge", has("posedge").cast("int"))
    .withColumn("has_negedge", has("negedge").cast("int"))
)

# -----------------------------------------------------------------------------
# Low-level operator / expression features
# -----------------------------------------------------------------------------

kg_feat = (kg_feat
    .withColumn("has_not_op", (
        (instr(col("code_no_comments"), "~") > 0)
    ).cast("int"))
    .withColumn("has_and_op", (
        (instr(col("code_no_comments"), "&") > 0) |
        (instr(col("code_no_comments"), "&&") > 0)
    ).cast("int"))
    .withColumn("has_or_op", (
        (instr(col("code_no_comments"), "|") > 0) |
        (instr(col("code_no_comments"), "||") > 0)
    ).cast("int"))
    .withColumn("has_xor_op", (
        (instr(col("code_no_comments"), "^") > 0)
    ).cast("int"))
    .withColumn("has_add_op", (
        (instr(col("code_no_comments"), "+") > 0)
    ).cast("int"))
    .withColumn("has_sub_op", (
        (instr(col("code_no_comments"), "-") > 0)
    ).cast("int"))
    .withColumn("has_lshift_op", (
        (instr(col("code_no_comments"), "<<") > 0)
    ).cast("int"))
    .withColumn("has_rshift_op", (
        (instr(col("code_no_comments"), ">>") > 0)
    ).cast("int"))
    .withColumn("has_concat", (
        regexp_extract(col("code_no_comments"), r"\{[^{}]+\}", 0) != ""
    ).cast("int"))
    .withColumn("has_compare_eq", (
        (instr(col("code_no_comments"), "==") > 0) |
        (instr(col("code_no_comments"), "===") > 0)
    ).cast("int"))
    .withColumn("has_compare_lt", (
        (instr(col("code_no_comments"), "<") > 0)
    ).cast("int"))
    .withColumn("has_compare_gt", (
        (instr(col("code_no_comments"), ">") > 0)
    ).cast("int"))
)

# -----------------------------------------------------------------------------
# Existing coarse tags + upgraded functional tags
# -----------------------------------------------------------------------------

kg_feat = (kg_feat
    .withColumn("tag_fsm", (
        (instr(col("code_no_comments"), "state") > 0) &
        (instr(col("code_no_comments"), "case") > 0)
    ).cast("int"))
    .withColumn("tag_counter", (
        (instr(col("code_no_comments"), "+ 1") > 0) |
        (instr(col("code_no_comments"), "++") > 0) |
        (instr(col("code_no_comments"), "counter") > 0)
    ).cast("int"))
    .withColumn("tag_mux", (
        (instr(col("code_no_comments"), "case") > 0) |
        (instr(col("code_no_comments"), "?:") > 0)
    ).cast("int"))
    .withColumn("tag_alu", (
        (instr(col("code_no_comments"), "alu") > 0) |
        ((col("has_add_op") == 1) & (col("has_sub_op") == 1))
    ).cast("int"))
    .withColumn("tag_shift", (
        (col("has_lshift_op") == 1) |
        (col("has_rshift_op") == 1) |
        (instr(col("code_no_comments"), "shift") > 0)
    ).cast("int"))
)

# -----------------------------------------------------------------------------
# Memory-related features
# -----------------------------------------------------------------------------

kg_feat = kg_feat.withColumn(
    "tag_mem_array",
    (regexp_extract(col("VerilogCode"), r"(?is)\breg\b[^\n;]*\[[^\]]+\]\s*\w+\s*\[[^\]]+\]\s*;", 0) != "").cast("int")
)

kg_feat = (kg_feat
    .withColumn("tag_ram", (
        (instr(col("code_no_comments"), "ram") > 0) |
        (instr(col("code_no_comments"), "memory") > 0) |
        (col("tag_mem_array") == 1)
    ).cast("int"))
    .withColumn("tag_fifo", (
        (instr(col("code_no_comments"), "fifo") > 0) |
        ((instr(col("code_no_comments"), "wr_ptr") > 0) & (instr(col("code_no_comments"), "rd_ptr") > 0)) |
        ((instr(col("code_no_comments"), "write_ptr") > 0) & (instr(col("code_no_comments"), "read_ptr") > 0))
    ).cast("int"))
)

# -----------------------------------------------------------------------------
# Clock / reset / sequential style
# -----------------------------------------------------------------------------

kg_feat = (kg_feat
    .withColumn("has_clk_signal", (
        (instr(col("code_no_comments"), " clk") > 0) |
        (instr(col("code_no_comments"), "(clk") > 0) |
        (instr(col("code_no_comments"), "clock") > 0)
    ).cast("int"))
    .withColumn("has_reset_signal", (
        (instr(col("code_no_comments"), "reset") > 0) |
        (instr(col("code_no_comments"), "rst") > 0)
    ).cast("int"))
    .withColumn("tag_async_reset", (
        (instr(col("code_no_comments"), "negedge rst") > 0) |
        (instr(col("code_no_comments"), "negedge reset") > 0) |
        (instr(col("code_no_comments"), "posedge reset") > 0) |
        (instr(col("code_no_comments"), "or negedge") > 0) |
        (instr(col("code_no_comments"), "or posedge reset") > 0)
    ).cast("int"))
)

# -----------------------------------------------------------------------------
# New functional tags
# -----------------------------------------------------------------------------

kg_feat = (kg_feat
    .withColumn("tag_constant_output", (
        (regexp_extract(col("code_no_comments"), r"assign\s+\w+\s*=\s*(1'b0|1'b1|0|1)\s*;", 0) != "") |
        (regexp_extract(col("code_no_comments"), r"\w+\s*<=\s*(1'b0|1'b1|0|1)\s*;", 0) != "")
    ).cast("int"))
    .withColumn("tag_not_gate", (
        (col("has_not_op") == 1) &
        (col("has_assign") == 1)
    ).cast("int"))
    .withColumn("tag_encoder", (
        ((instr(col("code_no_comments"), "encoder") > 0) |
         (instr(col("code_no_comments"), "priority") > 0)) &
        ((col("has_if") == 1) | (col("has_case") == 1))
    ).cast("int"))
    .withColumn("tag_decoder", (
        (instr(col("code_no_comments"), "decoder") > 0)
    ).cast("int"))
    .withColumn("tag_comparator", (
        (instr(col("code_no_comments"), "compare") > 0) |
        (col("has_compare_eq") == 1) |
        (col("has_compare_lt") == 1) |
        (col("has_compare_gt") == 1)
    ).cast("int"))
    .withColumn("tag_edge_detector", (
        (instr(col("code_no_comments"), "edge") > 0) |
        ((instr(col("code_no_comments"), "prev_") > 0) & (col("has_clk_signal") == 1)) |
        ((instr(col("code_no_comments"), "prev") > 0) & (instr(col("code_no_comments"), "^") > 0))
    ).cast("int"))
    .withColumn("tag_byte_reorder", (
        (regexp_extract(
            col("code_no_comments"),
            r"\{\s*\w+\[\d+:\d+\]\s*,\s*\w+\[\d+:\d+\]\s*,\s*\w+\[\d+:\d+\]\s*,\s*\w+\[\d+:\d+\]\s*\}",
            0
        ) != "")
    ).cast("int"))
)

# -----------------------------------------------------------------------------
# Instantiation / wrapper style
# -----------------------------------------------------------------------------

kg_feat = (kg_feat
    .withColumn("has_instantiation", (
        regexp_extract(col("VerilogCode"), r"(?im)^\s*[A-Za-z_]\w*\s+[A-Za-z_]\w*\s*\(", 0) != ""
    ).cast("int"))
    .withColumn("tag_wrapper_module", (
        (col("has_instantiation") == 1) &
        (col("has_assign") == 0) &
        (col("has_always") == 0)
    ).cast("int"))
)

# -----------------------------------------------------------------------------
# Interface / declaration shape features
# -----------------------------------------------------------------------------

# Count input/output declarations (approximate but useful)
kg_feat = (kg_feat
    .withColumn(
        "num_inputs",
        size(split(regexp_replace(regexp_extract(col("VerilogCode"), r"(?is)(.*)", 1), r"(?i)\binput\b", "§input§"), "§input§")) - 1
    )
    .withColumn(
        "num_outputs",
        size(split(regexp_replace(regexp_extract(col("VerilogCode"), r"(?is)(.*)", 1), r"(?i)\boutput\b", "§output§"), "§output§")) - 1
    )
)

# Extract up to a few declaration widths and take a max approximation
kg_feat = (kg_feat
    .withColumn("decl_w1", regexp_extract(col("VerilogCode"), r"\[(\d+)\s*:\s*0\]", 1))
    .withColumn("decl_w2", regexp_extract(col("VerilogCode"), r"\[\s*0\s*:\s*(\d+)\]", 1))
)

kg_feat = (kg_feat
    .withColumn(
        "max_decl_width",
        greatest(
            when(col("decl_w1") != "", col("decl_w1").cast("int") + lit(1)).otherwise(lit(1)),
            when(col("decl_w2") != "", col("decl_w2").cast("int") + lit(1)).otherwise(lit(1))
        )
    )
    .withColumn("has_wide_port_8plus", (col("max_decl_width") >= 8).cast("int"))
    .withColumn("has_wide_port_32plus", (col("max_decl_width") >= 32).cast("int"))
    .withColumn("is_single_output", (col("num_outputs") == 1).cast("int"))
)

# -----------------------------------------------------------------------------
# Assign-only / simple combinational shape
# -----------------------------------------------------------------------------

kg_feat = (kg_feat
    .withColumn("is_assign_only", (
        (col("has_assign") == 1) &
        (col("has_always") == 0) &
        (col("has_instantiation") == 0)
    ).cast("int"))
)

# -----------------------------------------------------------------------------
# Final selected table
# -----------------------------------------------------------------------------

kg_table = kg_feat.select(
    "doc_id",
    "module_name",

    # original structural features
    "has_always", "has_assign", "has_case", "has_if", "has_for", "has_generate",
    "has_posedge", "has_negedge",
    "has_clk_signal", "has_reset_signal", "tag_async_reset",
    "tag_fsm", "tag_counter", "tag_mux", "tag_alu", "tag_shift",
    "tag_mem_array", "tag_ram", "tag_fifo",

    # new low-level operator features
    "has_not_op", "has_and_op", "has_or_op", "has_xor_op",
    "has_add_op", "has_sub_op", "has_lshift_op", "has_rshift_op",
    "has_concat", "has_compare_eq", "has_compare_lt", "has_compare_gt",

    # new functional tags
    "tag_constant_output", "tag_not_gate", "tag_encoder", "tag_decoder",
    "tag_comparator", "tag_edge_detector", "tag_byte_reorder",
    "has_instantiation", "tag_wrapper_module",

    # interface / shape
    "num_inputs", "num_outputs", "max_decl_width",
    "has_wide_port_8plus", "has_wide_port_32plus", "is_single_output",
    "is_assign_only"
)

display(kg_table.limit(20))

In [0]:
kg_table.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("resyn27k_kg_features")

kg_features = spark.table("resyn27k_kg_features")
display(kg_features.limit(10))

kg_features.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("resyn27k_kg_features")

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.export_volume")
print("Volume ready: workspace.default.export_volume")

kg_df = spark.table("resyn27k_kg_features")

kg_output_path = "/Volumes/workspace/default/export_volume/kg_features_parquet"

(
    kg_df
    .coalesce(1)   # remove this line if the table is large
    .write
    .mode("overwrite")
    .parquet(kg_output_path)
)

print(f"KG parquet written to: {kg_output_path}")